1. **Nome(s):** Leonardo Silveira Kiepper

2. **Matrícula(s):** 2026140417

3. **E-mail(s) institucional(is):** leonardo.kiepper@edu.ufes.br

3. **Curso(s):** Doutorado em Ciência da Computação

4. **Link do seu Colab com suas soluções (para que o professor possa acessar e rodar o código):** https://colab.research.google.com/github/LeoKiepper/Ciencia-de-Dados/blob/main/trabalho1/T1_leonardo_kiepper.ipynb


In [1]:
%pip install itables
%pip install nltk
%pip install unidecode
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd # prática comum
from itables import show
from datetime import datetime
import nltk
#nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import sent_tokenize, word_tokenize
import unidecode
import string
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer


def descreve_dataset(df: pd.DataFrame):
    print(f"Número de livros: {len(df)}")
    print('')
    print(f"Quantos NaN em cada coluna: \n{df.isna().sum()}")
    print('')
    print("Autores distintos:", df['author'].nunique())
    # print(livros['author'].value_counts())    


[nltk_data] Downloading package punkt_tab to /home/Leo/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/Leo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


`book_id` não contribuirá para exprimir uma relação com `genre`, pois é apenas um indexador no *dataset*, e não ‘promete’ nenhuma relação com o conteúdo do livro. Qualquer relação encontrada seria expúria, então será usado como *index* do DataFrame.

In [3]:
# Importa o dataset completo

livros = pd.read_csv('goodreads_books.csv').set_index('book_id')
descreve_dataset(livros)

Número de livros: 25000

Quantos NaN em cada coluna: 
title                  0
author                 0
description            0
genre                  0
publication_year    5069
average_rating         0
ratings_count          0
num_pages           7273
dtype: int64

Autores distintos: 17509


In [4]:
show(livros.loc[:,livros.columns!='description'])

In [ ]:
# semente = np.random.randint(0, 2**31)
# df=livros.sample(n=100, random_state=semente)

# A título de ensaio, pega uma fatia pequena, e fixa a semente com random_state
semente = 2003343720
# semente = np.random.randint(0, 2**31)
df=livros.sample(n=100, random_state=semente)
descreve_dataset(df)
show(df)

Número de livros: 25000

Quantos NaN em cada coluna: 
title                  0
author                 0
description            0
genre                  0
publication_year    5069
average_rating         0
ratings_count          0
num_pages           7273
dtype: int64

Autores distintos: 17509


Aqui pode ser relevante fazer o pré-processamento para o titulo e para a descrição. A descrição "representa" o corpo do texto, e o titulo "resume" o livro. O titulo e a descrição carregam informações diferentes, e tem utilidades diferentes.

Sobre pré-processamento textual, do material de aula:
>É uma boa prática seguir, ao menos, os seguintes passos:
>1. Dividir o texto em *tokens*
>2. Converter o texto para letras minúsculas
>3. Remover símbolos de pontuação de cada *token*
>4. Converter caracteres especiais (passo discutível e necessidade deve ser analisada)
>5. Remover *tokens* que não são palavras
>6. Remover *tokens* que são *stop words*


In [6]:

def preprocessamento(df: pd.Series):
    # 1. Dividir o texto em tokens
    tokens = word_tokenize(df)  
    # tokens = sent_tokenize(df)

    # 2. Converter o texto para letras minúsculas
    tokens = [w.lower() for w in tokens]

    # 3. Remover símbolos de pontuação de cada token
    table = str.maketrans('', '', string.punctuation)
    tokens = [w.translate(table) for w in tokens]

    # 4. Converter caracteres especiais
    # tokens = [unidecode.unidecode(w).lower() for w in tokens]

    # 5. Remover tokens que não são palavras
    words = [word for word in tokens if word.isalpha()]

    # 6. Remover tokens que são stop words
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if not word in stop_words]
    
    # Stemming
    # porter = PorterStemmer()
    # stemmed = [porter.stem(word) for word in words]
    # words = stemmed
    
    return words

show(df['description'].apply(preprocessamento))

In [ ]:
def calculaX(x: pd.Series):   
    vectorizer = TfidfVectorizer(analyzer=preprocessamento)
    X = vectorizer.fit_transform(x)
    return pd.DataFrame(X, columns = vectorizer.get_feature_names_out())

X_desc = calculaX(df['description'])
show(X_desc)

X_title = calculaX(df['title'])
show(X_title)

In [ ]:
def encontra_token_max(X: pd.DataFrame) -> str:
    return X.max(axis=0).idxmax()
print(f"Tamanho do corpus, descrição: {len(X_desc.columns)}.")
print(f"\tTF-IDF máximo, descrição: {X_desc.max().max()}")
print(f"\ttoken com TF-IDF máximo: {encontra_token_max(X_desc)}")
print(f"Tamanho do corpus, título: {len(X_title.columns)}.")
print(f"\tTF-IDF máximo, título: {X_title.max().max()}")
print(f"\ttoken com TF-IDF máximo: {encontra_token_max(X_title)}")


Tamanho do corpus, descrição: 112003.
	TF-IDF máximo, descrição: 0.9311781254312337
	token com TF-IDF máximo: waldo
Tamanho do corpus, título: 19157.
	TF-IDF máximo, título: 1.0
	token com TF-IDF máximo: abandon


A estratégia de vetorizar pelo processo codificado nas células acima não é aplicável a todas as colunas do *dataset*. Por exemplo, a coluna Autor pode ser vista como um dado categórico, e precisaria de um tratamento diferente. 

As células acimas, responsáveis por vetorizar a descrição e o título, foram rodadas para o *dataset* completo para saber a dimensão máxima da matriz X:

>```
>Tamanho do corpus, descrição: 112003.
>Tamanho do corpus, título: 19157.
>```

Em comparação, as demais características presentes no *dataset*, descontando o `genre`, `description`, `title` e `book_id`, somam 5 dimensões à análise sobre o *dataset*. Dito isso, não faz sentido sujeitar essas dimensões ao algorítimo de PCA, junto com as matrizes `X`. O ganho seria comparativamente desprezível, e poderia haver perda de informação desproporcional.

Portanto, deseja-se fazer uma redução de dimensionalidade sobre `X_desc` e `X_title` separadamente. Desta forma, além de reduzir a dimensão tão somente sobre o que inviabiliza a análise, o algorítimo trabalha sobre dados de mesma natureza — e automaticamente na mesma escala — e assim, a perda de intuição sobre as grandezas dos dados é menos problemática. 